# 세션 검색

이 노트북에서는 오프라인 평가를 위해 AgentCore Observability에서 에이전트 세션을 검색합니다. 에이전트의 trace log group을 쿼리해 세션을 찾은 다음, 분석 노트북에서 처리할 수 있도록 JSON 파일에 저장합니다.

**두 가지 탐색 방법:**

1. **시간 기반**: 지정된 기간의 모든 세션을 찾습니다. 최근 에이전트 활동을 일괄 평가할 때 사용합니다.

2. **점수 기반**: AgentCore의 기존 평가 점수를 기준으로 세션을 찾습니다. 점수가 낮은 세션을 업데이트된 rubric으로 다시 평가할 때 사용합니다.

**출력:** 분석 노트북에서 사용할 세션 ID와 metadata가 포함된 `discovered_sessions.json`

## 전체 흐름에서의 위치

이 노트북은 평가 워크플로의 **노트북 1**입니다. 여기에서 세션을 검색한 뒤 두 가지 평가 경로 중 하나를 선택합니다.

![노트북 워크플로](images/notebook_workflow.svg)

## 설정

필요한 module을 import하고 `config.py`에서 구성을 불러옵니다. 이 cell을 실행하기 전에 환경 변수를 사용해 모든 구성 값을 재정의할 수 있습니다.

In [ ]:
import logging
import sys
from datetime import datetime, timedelta, timezone

sys.path.insert(0, ".")

from config import (
    AWS_REGION,
    SOURCE_LOG_GROUP,
    EVAL_RESULTS_LOG_GROUP_FULL,
    LOOKBACK_HOURS,
    MAX_SESSIONS,
    MIN_SCORE,
    MAX_SCORE,
    DISCOVERED_SESSIONS_PATH,
    EVALUATOR_NAME,
)

from utils import (
    ObservabilityClient,
    SessionDiscoveryResult,
    # SessionInfo,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

## 구성

점수 기반 검색에 사용하는 `EVALUATOR_NAME`은 `config.py`에서 불러옵니다. 이 값은 기존 평가 결과의 evaluator 이름과 일치해야 합니다. `LOOKBACK_HOURS`, `MAX_SESSIONS`, 점수 임계값 등의 설정을 변경하려면 `config.py`를 수정합니다.

In [ ]:
# EVALUATOR_NAME은 config.py에서 불러옵니다.
print(f"Using evaluator: {EVALUATOR_NAME}")

## 클라이언트 초기화

CloudWatch Logs Insights 쿼리를 처리하는 `ObservabilityClient`를 생성합니다. 시간 범위는 구성의 `LOOKBACK_HOURS`를 기준으로 계산합니다.

In [ ]:
obs_client = ObservabilityClient(
    region_name=AWS_REGION,
    log_group=SOURCE_LOG_GROUP,
)

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=LOOKBACK_HOURS)
start_time_ms = int(start_time.timestamp() * 1000)
end_time_ms = int(end_time.timestamp() * 1000)

## 시간 기반 검색

지정된 기간에 포함된 모든 고유 세션 ID를 찾도록 AgentCore Observability log group을 쿼리합니다. 최근 활동 순으로 정렬된 세션과 span 수, timestamp를 반환합니다. 최근의 모든 에이전트 상호작용을 평가하려면 이 방법을 사용합니다.

In [ ]:
time_based_sessions = obs_client.discover_sessions(
    start_time_ms=start_time_ms,
    end_time_ms=end_time_ms,
    limit=MAX_SESSIONS,
)

print(f"Discovered {len(time_based_sessions)} sessions")

## 점수 기반 검색

기존 평가 점수를 기준으로 세션을 찾도록 AgentCore Evaluations 결과 log group을 쿼리합니다. 지정한 evaluator의 점수가 `MIN_SCORE`와 `MAX_SCORE` 사이인 세션만 필터링합니다. 성능이 낮은 세션을 찾아 업데이트된 rubric으로 다시 평가하려면 이 방법을 사용합니다.

In [ ]:
score_based_sessions = obs_client.discover_sessions_by_score(
    evaluation_log_group=EVAL_RESULTS_LOG_GROUP_FULL,
    evaluator_name=EVALUATOR_NAME,
    start_time_ms=start_time_ms,
    end_time_ms=end_time_ms,
    min_score=MIN_SCORE,
    max_score=MAX_SCORE,
    limit=MAX_SESSIONS,
)

print(f"Discovered {len(score_based_sessions)} sessions by score")

## 검색 방법 선택

검색된 세션 중 사용할 세트를 선택합니다. 시간 기반 결과를 사용하려면 `USE_TIME_BASED = True`로, 점수 기반 결과를 사용하려면 `False`로 설정합니다. 선택한 세션은 검색 방법에 대한 metadata와 함께 `SessionDiscoveryResult`로 구성됩니다.

In [ ]:
# 점수 기반 검색을 사용하려면 False로 설정합니다.
USE_TIME_BASED = True

if USE_TIME_BASED:
    selected_sessions = time_based_sessions
    discovery_method = "time_based"
    log_group = SOURCE_LOG_GROUP
    filter_criteria = None
else:
    selected_sessions = score_based_sessions
    discovery_method = "score_based"
    log_group = EVAL_RESULTS_LOG_GROUP_FULL
    filter_criteria = {
        "evaluator_name": EVALUATOR_NAME,
        "min_score": MIN_SCORE,
        "max_score": MAX_SCORE,
    }

discovery_result = SessionDiscoveryResult(
    sessions=selected_sessions,
    discovery_time=datetime.now(timezone.utc),
    log_group=log_group,
    time_range_start=start_time,
    time_range_end=end_time,
    discovery_method=discovery_method,
    filter_criteria=filter_criteria,
)

print(f"Selected {len(selected_sessions)} sessions via {discovery_method}")

## 세션 미리 보기

검색된 세션 중 처음 10개를 확인합니다. 시간 기반 검색에서는 세션 ID와 span 수를, 점수 기반 검색에서는 세션 ID와 평균 평가 점수를 표시합니다.

In [ ]:
for i, session in enumerate(selected_sessions[:10]):
    meta = session.metadata or {}
    if discovery_method == "time_based":
        print(f"{i + 1}. {session.session_id} - {session.span_count} spans")
    else:
        print(f"{i + 1}. {session.session_id} - avg_score: {meta.get('avg_score', 0):.2f}")

## 결과 저장

검색 결과를 JSON으로 저장합니다. 분석 노트북은 이 파일을 불러와 각 세션을 처리합니다. 출력 경로는 `config.py`에 `DISCOVERED_SESSIONS_PATH`로 구성되어 있습니다.

In [ ]:
discovery_result.save_to_json(DISCOVERED_SESSIONS_PATH)
print(f"Saved {len(selected_sessions)} sessions to {DISCOVERED_SESSIONS_PATH}")

## 출력 확인

JSON 파일이 올바르게 저장되었는지 확인합니다. 확인을 마치면 다중 세션 분석 노트북으로 이동해 이 세션들을 평가합니다.

In [ ]:
import json

with open(DISCOVERED_SESSIONS_PATH, "r") as f:
    saved_data = json.load(f)

print(f"Sessions: {len(saved_data['sessions'])}")
print(f"Method: {saved_data['discovery_method']}")
print(f"Time range: {saved_data['time_range_start']} to {saved_data['time_range_end']}")